In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityStatevectorKernel
from qiskit_machine_learning.algorithms import QSVC

data = load_breast_cancer()
X, y = data.data, data.target
n_qubits = 4

In [2]:
x = ParameterVector("x", n_qubits)
simple_circuit = QuantumCircuit(n_qubits)
for i in range(n_qubits):
    simple_circuit.ry(x[i], i)

entangled_circuit = zz_feature_map(n_qubits, reps=2)

In [3]:
def run_once(seed):

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed
    )

    scaler = StandardScaler().fit(X_train)
    pca = PCA(n_components=n_qubits).fit(scaler.transform(X_train))
    to_angle = MinMaxScaler(feature_range=(0, np.pi)).fit(pca.transform(scaler.transform(X_train)))
    prepare = lambda A: to_angle.transform(pca.transform(scaler.transform(A)))
    X_train_q, X_test_q = prepare(X_train), prepare(X_test)

    models = {
        "Classical SVM": SVC(kernel="rbf"),
        "Quantum SVM (simple)": QSVC(quantum_kernel=FidelityStatevectorKernel(feature_map=simple_circuit)),
        "Quantum SVM (entangled)": QSVC(quantum_kernel=FidelityStatevectorKernel(feature_map=entangled_circuit)),
    }

    row = {"split": seed}
    for name, model in models.items():
        model.fit(X_train_q, y_train)
        row[name] = accuracy_score(y_test, model.predict(X_test_q))
    return row

In [4]:
results = pd.DataFrame([run_once(seed) for seed in range(10)])
print(results.round(3))

   split  Classical SVM  Quantum SVM (simple)  Quantum SVM (entangled)
0      0          0.930                 0.939                    0.781
1      1          0.956                 0.965                    0.816
2      2          0.974                 0.974                    0.798
3      3          0.930                 0.956                    0.807
4      4          0.974                 0.974                    0.737
5      5          0.974                 0.974                    0.781
6      6          0.974                 0.982                    0.798
7      7          0.947                 0.947                    0.746
8      8          0.974                 0.956                    0.807
9      9          0.982                 0.982                    0.798


In [5]:
summary = results.drop(columns="split").agg(["mean", "std"]).T
print(summary.round(3))

os.makedirs("../results", exist_ok=True)
results.to_csv("../results/exp1_breast_cancer_10_splits.csv", index=False)

                          mean    std
Classical SVM            0.961  0.019
Quantum SVM (simple)     0.965  0.015
Quantum SVM (entangled)  0.787  0.026
